# Notebook 2 — Sales EDA

**Goal:** Verify that the demand model's causal mechanisms are visible in the data.

We should observe:
- Weekend grocery sales > weekday
- Nov/Dec spike for Grocery and General Merchandise
- Holiday-day demand spikes
- Promotion lift followed by post-promo dip
- Weather effects on relevant categories

If you can't see these patterns, the data is not useful for ML. This EDA step is what separates a real data scientist from someone who just builds models.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from sqlalchemy import create_engine

DB_URL = os.getenv('SUPPLY_CHAIN_DB_URL', '')
engine = create_engine(DB_URL)
SCHEMA = 'supply_chain'

def q(sql):
    return pd.read_sql(sql, engine)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Ready.')

## 1. Weekly total revenue over time

This is your first sanity check: does the data look like real retail? You should see:
- Steady baseline
- Clear November–December peaks
- Super Bowl and Easter bumps

In [ ]:
weekly = q(f"""
SELECT
    DATE_TRUNC('week', date_id::timestamp)::date AS week_start,
    category,
    SUM(net_revenue) AS weekly_revenue
FROM {SCHEMA}.v_daily_store_sales
GROUP BY week_start, category
ORDER BY week_start, category
""")
weekly['week_start'] = pd.to_datetime(weekly['week_start'])

fig, ax = plt.subplots()
for cat in weekly['category'].unique():
    sub = weekly[weekly['category'] == cat]
    ax.plot(sub['week_start'], sub['weekly_revenue'] / 1e6, label=cat, linewidth=1.5)
ax.set_title('Weekly Revenue by Category ($M)')
ax.set_xlabel('Week')
ax.set_ylabel('Revenue ($M)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Day-of-week pattern (Grocery)

In real grocery retail, Saturday and Sunday are the highest-volume days. Weekday traffic is about 70–80% of weekend levels.

In [ ]:
dow = q(f"""
SELECT
    c.day_name,
    c.day_of_week,
    ROUND(AVG(ds.total_units_sold), 2) AS avg_units,
    ROUND(AVG(ds.total_net_revenue), 2) AS avg_revenue
FROM {SCHEMA}.v_daily_store_sales ds
JOIN {SCHEMA}.calendar c ON c.date_id = ds.date_id
WHERE ds.category = 'Grocery'
GROUP BY c.day_name, c.day_of_week
ORDER BY c.day_of_week
""")

fig, ax = plt.subplots()
colors = ['#2196F3' if dow_val < 6 else '#F44336' for dow_val in dow['day_of_week']]
ax.bar(dow['day_name'], dow['avg_units'], color=colors)
ax.set_title('Average Daily Units Sold — Grocery (per store-product)')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Avg Units')
plt.tight_layout()
plt.show()
print(dow[['day_name', 'avg_units', 'avg_revenue']])

## 3. Seasonal patterns by category (monthly avg)

Key things to verify:
- Grocery peaks Nov–Dec
- General Merchandise has the strongest holiday spike
- Health & Beauty peaks Jan–Feb (cold/flu season)

In [ ]:
monthly = q(f"""
SELECT
    c.month,
    c.month_name,
    ds.category,
    SUM(ds.total_units_sold) AS total_units,
    SUM(ds.total_net_revenue) AS total_revenue
FROM {SCHEMA}.v_daily_store_sales ds
JOIN {SCHEMA}.calendar c ON c.date_id = ds.date_id
GROUP BY c.month, c.month_name, ds.category
ORDER BY c.month, ds.category
""")

# Normalize to index (month 1 = 100) for each category
base = monthly.groupby('category')['total_units'].transform(lambda x: x.iloc[0])
monthly['indexed'] = monthly['total_units'] / base * 100

fig, ax = plt.subplots()
for cat in monthly['category'].unique():
    sub = monthly[monthly['category'] == cat]
    ax.plot(sub['month'], sub['indexed'], marker='o', label=cat)
ax.axhline(100, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Monthly Demand Index by Category (Jan = 100)')
ax.set_xlabel('Month')
ax.set_ylabel('Index')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend()
plt.tight_layout()
plt.show()

## 4. Holiday demand spikes

Compare sales on specific holidays vs. the same day-of-week in the surrounding weeks.

In [ ]:
holiday_lift = q(f"""
SELECT
    c.holiday_name,
    ds.category,
    ROUND(AVG(ds.total_units_sold), 2) AS holiday_avg_units
FROM {SCHEMA}.v_daily_store_sales ds
JOIN {SCHEMA}.calendar c ON c.date_id = ds.date_id
WHERE c.is_holiday = TRUE AND c.holiday_name IN (
    'Thanksgiving', 'Black Friday', 'Christmas Eve', 'Christmas Day',
    'Super Bowl Sunday', 'Independence Day', 'Halloween'
)
GROUP BY c.holiday_name, ds.category
ORDER BY ds.category, holiday_avg_units DESC
""")

baseline = q(f"""
SELECT category, ROUND(AVG(total_units_sold), 2) AS baseline_avg_units
FROM {SCHEMA}.v_daily_store_sales
GROUP BY category
""")

holiday_lift = holiday_lift.merge(baseline, on='category')
holiday_lift['lift_factor'] = (holiday_lift['holiday_avg_units'] / holiday_lift['baseline_avg_units']).round(2)

pivot = holiday_lift.pivot(index='holiday_name', columns='category', values='lift_factor')
print('Holiday demand lift vs. baseline average:')
print(pivot.round(2).to_string())

## 5. Top-selling products

The Pareto principle applies in retail: 20% of SKUs generate 80% of revenue. This is the foundation of ABC analysis (Project 1 extension).

In [ ]:
top_products = q(f"""
SELECT
    p.sku,
    p.product_name,
    p.category,
    SUM(st.quantity_sold) AS total_units,
    SUM(st.net_revenue) AS total_revenue,
    COUNT(DISTINCT st.store_id) AS stores_selling
FROM {SCHEMA}.sales_transactions st
JOIN {SCHEMA}.products p ON p.product_id = st.product_id
GROUP BY p.sku, p.product_name, p.category
ORDER BY total_revenue DESC
LIMIT 20
""")
top_products['total_revenue'] = top_products['total_revenue'].apply(lambda x: f'${x:,.0f}')
top_products['total_units']   = top_products['total_units'].apply(lambda x: f'{x:,}')
print(top_products.to_string(index=False))

## 6. ABC Analysis — the foundation of SKU segmentation

In supply chain, SKUs are classified:
- **A**: Top 80% of revenue — highest service level, tightest inventory control
- **B**: Next 15% — moderate attention
- **C**: Bottom 5% — minimal inventory, consider delisting

This classification changes your entire replenishment strategy.

In [ ]:
sku_revenue = q(f"""
SELECT product_id, SUM(net_revenue) AS total_revenue
FROM {SCHEMA}.sales_transactions
GROUP BY product_id
ORDER BY total_revenue DESC
""")

sku_revenue['cum_revenue'] = sku_revenue['total_revenue'].cumsum()
sku_revenue['cum_pct']     = sku_revenue['cum_revenue'] / sku_revenue['total_revenue'].sum()
sku_revenue['abc_class']   = sku_revenue['cum_pct'].apply(
    lambda x: 'A' if x <= 0.80 else ('B' if x <= 0.95 else 'C')
)

abc_summary = sku_revenue.groupby('abc_class').agg(
    sku_count=('product_id', 'count'),
    total_revenue=('total_revenue', 'sum')
).reset_index()
abc_summary['pct_skus']    = (abc_summary['sku_count'] / len(sku_revenue) * 100).round(1)
abc_summary['pct_revenue'] = (abc_summary['total_revenue'] / sku_revenue['total_revenue'].sum() * 100).round(1)

print('ABC Analysis:')
print(abc_summary[['abc_class', 'sku_count', 'pct_skus', 'pct_revenue']].to_string(index=False))

# Pareto chart
fig, ax1 = plt.subplots()
n = min(500, len(sku_revenue))
ax1.bar(range(n), sku_revenue['total_revenue'].head(n) / 1000, color='steelblue', alpha=0.6)
ax2 = ax1.twinx()
ax2.plot(range(n), sku_revenue['cum_pct'].head(n) * 100, color='red', linewidth=1.5)
ax2.axhline(80, color='gray', linestyle='--', linewidth=0.8, label='80%')
ax1.set_title('Pareto Chart: SKU Revenue (top 500 SKUs)')
ax1.set_xlabel('SKU rank')
ax1.set_ylabel('Revenue ($K)')
ax2.set_ylabel('Cumulative %')
plt.tight_layout()
plt.show()

## 7. Weather effect on Grocery

Does bad weather actually reduce store traffic? Does hot weather boost beverage sales?

In [ ]:
weather_effect = q(f"""
SELECT
    w.weather_severity,
    ds.category,
    ROUND(AVG(ds.total_units_sold), 3) AS avg_units
FROM {SCHEMA}.v_daily_store_sales ds
JOIN {SCHEMA}.weather w ON w.store_id = ds.store_id AND w.date_id = ds.date_id
GROUP BY w.weather_severity, ds.category
ORDER BY ds.category, w.weather_severity
""")

weather_pivot = weather_effect.pivot(index='weather_severity', columns='category', values='avg_units')
print('Average daily units by weather severity:')
print(weather_pivot.round(3).to_string())